# Treinamento NLP multirrótulo — detecção textual de risco

Este notebook treina um classificador para cinco rótulos:

- `pedido_socorro`
- `ameaca`
- `agressao`
- `referencia_arma`
- `nao_violento`

Uma frase pode possuir mais de um rótulo de risco. Por isso, o problema é de **classificação multirrótulo**, usando:

- cinco saídas com ativação sigmoide;
- `BCEWithLogitsLoss`;
- pesos positivos calculados a partir do conjunto de treino;
- métricas micro, macro e por classe;
- ajuste dos limiares usando apenas a validação.


In [ ]:
!pip -q install     "transformers==4.57.6"     "datasets>=3.2,<5"     "accelerate>=1.2,<2"     "scikit-learn>=1.5"     "pandas>=2.2"     "matplotlib>=3.8"     "tqdm>=4.66"     "safetensors>=0.4"

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import random
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: Falha ao usar GPU")

## 1. Configuração


In [ ]:
@dataclass(frozen=True)
class Configuracao:
    modelo_base: str = "neuralmind/bert-base-portuguese-cased"
    comprimento_maximo: int = 96

    epocas: int = 4
    lote_treino: int = 16
    lote_avaliacao: int = 32
    acumulacao_gradientes: int = 1

    taxa_aprendizado: float = 2e-5
    decaimento_peso: float = 0.01
    aquecimento: float = 0.10

    paciencia: int = 2
    passos_log: int = 50
    semente: int = 42

    limiar_inicial: float = 0.50
    limiar_minimo: float = 0.10
    limiar_maximo: float = 0.90
    passo_limiar: float = 0.01


configuracao = Configuracao()

ROTULOS = [
    "pedido_socorro",
    "ameaca",
    "agressao",
    "referencia_arma",
    "nao_violento",
]

ID_PARA_ROTULO = {
    indice: rotulo
    for indice, rotulo in enumerate(ROTULOS)
}

ROTULO_PARA_ID = {
    rotulo: indice
    for indice, rotulo in ID_PARA_ROTULO.items()
}

PASTA_RAIZ = (
    Path.home()
    / "Documentos"
    / "projetos"
    / "violence-edge-ai"
)

PASTA_DADOS = (
    PASTA_RAIZ
    / "datasets"
    / "nlp"
    / "splits"
)

PASTA_CHECKPOINTS = (
    PASTA_RAIZ
    / "models"
    / "nlp"
    / "checkpoints"
)

PASTA_MODELO = (
    PASTA_RAIZ
    / "models"
    / "nlp"
    / "bertimbau_multirrotulo"
)

PASTA_METRICAS = (
    PASTA_RAIZ
    / "outputs"
    / "metrics"
    / "nlp"
)

for pasta in (
    PASTA_CHECKPOINTS,
    PASTA_MODELO,
    PASTA_METRICAS,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(configuracao.semente)
np.random.seed(configuracao.semente)
torch.manual_seed(configuracao.semente)
set_seed(configuracao.semente)

display(pd.DataFrame([asdict(configuracao)]))

## 2. Enviar o dataset

Envie os três arquivos:

- `treino.jsonl`
- `validacao.jsonl`
- `teste.jsonl`


In [ ]:
from google.colab import files
import zipfile

arquivos_esperados = {
    "treino.jsonl",
    "validacao.jsonl",
    "teste.jsonl",
}

arquivos_existentes = {
    caminho.name
    for caminho in PASTA_DADOS.rglob("*.jsonl")
}

if not arquivos_esperados.issubset(arquivos_existentes):
    enviados = files.upload()

    for nome, conteudo in enviados.items():
        destino = PASTA_DADOS / nome
        destino.write_bytes(conteudo)

        if nome.lower().endswith(".zip"):
            with zipfile.ZipFile(destino) as arquivo_zip:
                arquivo_zip.extractall(PASTA_DADOS)

print("Arquivos encontrados:")

for nome in sorted(arquivos_esperados):
    encontrados = list(PASTA_DADOS.rglob(nome))

    if not encontrados:
        raise FileNotFoundError(
            f"O arquivo {nome} não foi encontrado."
        )

    print(nome, "→", encontrados[0])

## 3. Carregar e validar os arquivos JSONL

In [ ]:
def localizar_arquivo(nome: str) -> Path:
    encontrados = list(PASTA_DADOS.rglob(nome))

    if not encontrados:
        raise FileNotFoundError(
            f"Arquivo não encontrado: {nome}"
        )

    return encontrados[0]


def carregar_jsonl(
    caminho: Path,
    nome_split: str,
) -> pd.DataFrame:
    registros = []

    with caminho.open("r", encoding="utf-8") as arquivo:
        for numero_linha, linha in enumerate(
            tqdm(
                arquivo,
                desc=f"Carregando {nome_split}",
                unit="linha",
            ),
            start=1,
        ):
            linha = linha.strip()

            if not linha:
                continue

            try:
                registro = json.loads(linha)
            except json.JSONDecodeError as erro:
                raise ValueError(
                    f"JSON inválido em {caminho}, linha {numero_linha}."
                ) from erro

            registros.append(registro)

    tabela = pd.DataFrame(registros)

    campos_obrigatorios = {
        "id",
        "texto",
        "rotulos",
    }

    faltantes = campos_obrigatorios - set(tabela.columns)

    if faltantes:
        raise ValueError(
            f"Campos ausentes em {nome_split}: {faltantes}"
        )

    tabela["split"] = nome_split

    return tabela


tabelas = {
    "treino": carregar_jsonl(
        localizar_arquivo("treino.jsonl"),
        "treino",
    ),
    "validacao": carregar_jsonl(
        localizar_arquivo("validacao.jsonl"),
        "validacao",
    ),
    "teste": carregar_jsonl(
        localizar_arquivo("teste.jsonl"),
        "teste",
    ),
}

for nome, tabela in tabelas.items():
    print(
        nome,
        "→",
        len(tabela),
        "exemplos",
    )

display(tabelas["treino"].head())

## 4. Preparar os vetores multirrótulo

In [ ]:
def criar_vetor_rotulos(rotulos: dict) -> list[float]:
    if not isinstance(rotulos, dict):
        raise ValueError("O campo 'rotulos' precisa ser um dicionário.")

    vetor = []

    for nome in ROTULOS:
        valor = rotulos.get(nome)

        if valor not in (0, 1):
            raise ValueError(
                f"Rótulo inválido para {nome}: {valor}"
            )

        vetor.append(float(valor))

    return vetor


for nome, tabela in tabelas.items():
    tabela["labels"] = tabela["rotulos"].apply(
        criar_vetor_rotulos
    )

    matriz = np.asarray(
        tabela["labels"].tolist(),
        dtype=np.float32,
    )

    print(f"\n{nome}:")
    print(
        pd.Series(
            matriz.sum(axis=0).astype(int),
            index=ROTULOS,
        )
    )

## 5. Calcular os pesos positivos

Os pesos reduzem o efeito do desbalanceamento entre os rótulos.

Para cada rótulo:

```text
peso positivo = quantidade de negativos / quantidade de positivos
```

In [ ]:
rotulos_treino = np.asarray(
    tabelas["treino"]["labels"].tolist(),
    dtype=np.float32,
)

positivos = rotulos_treino.sum(axis=0)
negativos = len(rotulos_treino) - positivos

pesos_positivos = negativos / np.maximum(
    positivos,
    1,
)

tabela_pesos = pd.DataFrame(
    {
        "rotulo": ROTULOS,
        "positivos": positivos.astype(int),
        "negativos": negativos.astype(int),
        "peso_positivo": pesos_positivos,
    }
)

display(tabela_pesos)

tensor_pesos_positivos = torch.tensor(
    pesos_positivos,
    dtype=torch.float32,
)

## 6. Criar datasets e tokenizar

In [ ]:
tokenizador = AutoTokenizer.from_pretrained(
    configuracao.modelo_base,
    do_lower_case=False,
)


def criar_dataset(tabela: pd.DataFrame) -> Dataset:
    dados = tabela[
        [
            "texto",
            "labels",
        ]
    ].copy()

    return Dataset.from_pandas(
        dados,
        preserve_index=False,
    )


datasets = {
    nome: criar_dataset(tabela)
    for nome, tabela in tabelas.items()
}


def tokenizar_lote(lote: dict) -> dict:
    return tokenizador(
        lote["texto"],
        truncation=True,
        max_length=configuracao.comprimento_maximo,
    )


datasets_tokenizados = {
    nome: dataset.map(
        tokenizar_lote,
        batched=True,
        desc=f"Tokenizando {nome}",
    )
    for nome, dataset in datasets.items()
}

colator = DataCollatorWithPadding(
    tokenizer=tokenizador,
)

display(datasets_tokenizados["treino"][0])

## 7. Criar modelo e métricas

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    configuracao.modelo_base,
    num_labels=len(ROTULOS),
    id2label=ID_PARA_ROTULO,
    label2id=ROTULO_PARA_ID,
    problem_type="multi_label_classification",
)

print(
    "Parâmetros:",
    f"{modelo.num_parameters():,}".replace(",", "."),
)

In [ ]:
def sigmoid(logits: np.ndarray) -> np.ndarray:
    logits = np.clip(logits, -30, 30)
    return 1.0 / (1.0 + np.exp(-logits))


def calcular_metricas(
    rotulos_reais: np.ndarray,
    probabilidades: np.ndarray,
    limiares,
) -> dict:
    limiares = np.asarray(limiares)
    previstos = (
        probabilidades >= limiares
    ).astype(int)

    metricas = {
        "f1_micro": f1_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            rotulos_reais,
            previstos,
            average="macro",
            zero_division=0,
        ),
        "f1_ponderado": f1_score(
            rotulos_reais,
            previstos,
            average="weighted",
            zero_division=0,
        ),
        "precisao_micro": precision_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "revocacao_micro": recall_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "acuracia_exata": accuracy_score(
            rotulos_reais,
            previstos,
        ),
        "hamming_loss": hamming_loss(
            rotulos_reais,
            previstos,
        ),
    }

    try:
        metricas["roc_auc_macro"] = roc_auc_score(
            rotulos_reais,
            probabilidades,
            average="macro",
        )
    except ValueError:
        metricas["roc_auc_macro"] = 0.0

    return {
        chave: float(valor)
        for chave, valor in metricas.items()
    }


def metricas_treinamento(predicao) -> dict:
    logits, rotulos_reais = predicao

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilidades = sigmoid(logits)

    return calcular_metricas(
        rotulos_reais.astype(int),
        probabilidades,
        configuracao.limiar_inicial,
    )

## 8. Treinador com perda ponderada

A classe abaixo altera apenas o cálculo da perda. O restante continua usando o `Trainer` padrão.

In [ ]:
class TreinadorMultirrotulo(Trainer):
    def __init__(
        self,
        pesos_positivos: torch.Tensor,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.pesos_positivos = pesos_positivos

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        rotulos = inputs.pop("labels").float()
        saidas = model(**inputs)
        logits = saidas.logits

        funcao_perda = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pesos_positivos.to(
                logits.device
            )
        )

        perda = funcao_perda(
            logits,
            rotulos,
        )

        if return_outputs:
            return perda, saidas

        return perda

## 9. Configurar e executar o treinamento

In [ ]:
argumentos_treinamento = TrainingArguments(
    output_dir=str(PASTA_SAIDA / "checkpoints"),
    num_train_epochs=configuracao.epocas,
    learning_rate=configuracao.taxa_aprendizado,
    per_device_train_batch_size=configuracao.lote_treino,
    per_device_eval_batch_size=configuracao.lote_avaliacao,
    gradient_accumulation_steps=configuracao.acumulacao_gradientes,
    weight_decay=configuracao.decaimento_peso,
    warmup_ratio=configuracao.aquecimento,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=configuracao.passos_log,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=configuracao.semente,
    data_seed=configuracao.semente,
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

treinador = TreinadorMultirrotulo(
    pesos_positivos=tensor_pesos_positivos,
    model=modelo,
    args=argumentos_treinamento,
    train_dataset=datasets_tokenizados["treino"],
    eval_dataset=datasets_tokenizados["validacao"],
    processing_class=tokenizador,
    data_collator=colator,
    compute_metrics=metricas_treinamento,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=configuracao.paciencia
        )
    ],
)

resultado_treinamento = treinador.train()

print("Treinamento concluído.")
print(resultado_treinamento.metrics)

## 10. Histórico e gráficos do treinamento

In [ ]:
historico = pd.DataFrame(
    treinador.state.log_history
)

display(historico)

historico.to_csv(
    PASTA_METRICAS / "historico_treinamento.csv",
    index=False,
)

In [ ]:
perdas_treino = historico[
    historico["loss"].notna()
][
    [
        "step",
        "loss",
    ]
]

plt.figure(figsize=(10, 5))
plt.plot(
    perdas_treino["step"],
    perdas_treino["loss"],
)
plt.xlabel("Passo")
plt.ylabel("Perda")
plt.title("Perda durante o treinamento")
plt.grid()
plt.show()

In [ ]:
avaliacoes = historico[
    historico["eval_loss"].notna()
].copy()

plt.figure(figsize=(8, 5))
plt.plot(
    avaliacoes["epoch"],
    avaliacoes["eval_loss"],
    marker="o",
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("Perda")
plt.title("Perda de validação")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    avaliacoes["epoch"],
    avaliacoes["eval_f1_micro"],
    marker="o",
    label="F1 micro",
)
plt.plot(
    avaliacoes["epoch"],
    avaliacoes["eval_f1_macro"],
    marker="o",
    label="F1 macro",
)
plt.xlabel("Época")
plt.ylabel("F1")
plt.title("F1 na validação")
plt.legend()
plt.grid()
plt.show()

## 11. Ajustar limiares na validação

Cada rótulo recebe um limiar próprio. O conjunto de teste não participa dessa escolha.

In [ ]:
predicao_validacao = treinador.predict(
    datasets_tokenizados["validacao"]
)

logits_validacao = predicao_validacao.predictions

if isinstance(logits_validacao, tuple):
    logits_validacao = logits_validacao[0]

probabilidades_validacao = sigmoid(
    logits_validacao
)

rotulos_validacao = np.asarray(
    tabelas["validacao"]["labels"].tolist(),
    dtype=int,
)


def encontrar_melhor_limiar(
    reais: np.ndarray,
    probabilidades: np.ndarray,
) -> tuple[float, float]:
    melhor_limiar = configuracao.limiar_inicial
    melhor_f1 = -1.0

    candidatos = np.arange(
        configuracao.limiar_minimo,
        configuracao.limiar_maximo
        + configuracao.passo_limiar,
        configuracao.passo_limiar,
    )

    for limiar in candidatos:
        previstos = (
            probabilidades >= limiar
        ).astype(int)

        f1 = f1_score(
            reais,
            previstos,
            zero_division=0,
        )

        if f1 > melhor_f1:
            melhor_f1 = f1
            melhor_limiar = float(limiar)

    return melhor_limiar, melhor_f1


limiares_otimizados = []
resultados_limiares = []

for indice, rotulo in enumerate(ROTULOS):
    limiar, f1 = encontrar_melhor_limiar(
        rotulos_validacao[:, indice],
        probabilidades_validacao[:, indice],
    )

    limiares_otimizados.append(limiar)

    resultados_limiares.append(
        {
            "rotulo": rotulo,
            "limiar": limiar,
            "f1_validacao": f1,
        }
    )

tabela_limiares = pd.DataFrame(
    resultados_limiares
)

display(tabela_limiares)

tabela_limiares.to_csv(
    PASTA_METRICAS / "limiares_otimizados.csv",
    index=False,
)

(PASTA_MODELO / "limiares.json").write_text(
    json.dumps(
        {
            rotulo: float(limiar)
            for rotulo, limiar in zip(
                ROTULOS,
                limiares_otimizados,
            )
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 12. Avaliar no conjunto de teste

In [ ]:
predicao_teste = treinador.predict(
    datasets_tokenizados["teste"]
)

logits_teste = predicao_teste.predictions

if isinstance(logits_teste, tuple):
    logits_teste = logits_teste[0]

probabilidades_teste = sigmoid(
    logits_teste
)

rotulos_teste = np.asarray(
    tabelas["teste"]["labels"].tolist(),
    dtype=int,
)

previstos_padrao = (
    probabilidades_teste
    >= configuracao.limiar_inicial
).astype(int)

previstos_otimizados = (
    probabilidades_teste
    >= np.asarray(limiares_otimizados)
).astype(int)

metricas_padrao = calcular_metricas(
    rotulos_teste,
    probabilidades_teste,
    configuracao.limiar_inicial,
)

metricas_otimizadas = calcular_metricas(
    rotulos_teste,
    probabilidades_teste,
    limiares_otimizados,
)

comparacao_metricas = pd.DataFrame(
    [
        {
            "configuracao": "limiar_0.50",
            **metricas_padrao,
        },
        {
            "configuracao": "limiares_otimizados",
            **metricas_otimizadas,
        },
    ]
)

display(comparacao_metricas)

## 13. Relatório por classe

In [ ]:
relatorio_classes = classification_report(
    rotulos_teste,
    previstos_otimizados,
    target_names=ROTULOS,
    output_dict=True,
    zero_division=0,
)

tabela_classes = pd.DataFrame(
    relatorio_classes
).transpose()

display(tabela_classes)

tabela_classes.to_csv(
    PASTA_METRICAS / "metricas_teste_por_classe.csv"
)

In [ ]:
f1_padrao = f1_score(
    rotulos_teste,
    previstos_padrao,
    average=None,
    zero_division=0,
)

f1_otimizado = f1_score(
    rotulos_teste,
    previstos_otimizados,
    average=None,
    zero_division=0,
)

posicoes = np.arange(len(ROTULOS))
largura = 0.35

plt.figure(figsize=(11, 5))
plt.bar(
    posicoes - largura / 2,
    f1_padrao,
    width=largura,
    label="limiar 0,50",
)
plt.bar(
    posicoes + largura / 2,
    f1_otimizado,
    width=largura,
    label="limiar otimizado",
)
plt.xticks(
    posicoes,
    ROTULOS,
    rotation=25,
)
plt.xlabel("Rótulo")
plt.ylabel("F1")
plt.title("F1 por rótulo no conjunto de teste")
plt.legend()
plt.grid(axis="y")
plt.show()

## 14. Matrizes de confusão por rótulo

In [ ]:
matrizes = multilabel_confusion_matrix(
    rotulos_teste,
    previstos_otimizados,
)

for indice, rotulo in enumerate(ROTULOS):
    matriz = matrizes[indice]

    plt.figure(figsize=(5, 5))
    plt.imshow(matriz)
    plt.xticks(
        [0, 1],
        ["previsto 0", "previsto 1"],
    )
    plt.yticks(
        [0, 1],
        ["real 0", "real 1"],
    )
    plt.title(f"Matriz de confusão — {rotulo}")
    plt.colorbar()

    for linha in range(2):
        for coluna in range(2):
            plt.text(
                coluna,
                linha,
                int(matriz[linha, coluna]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()

## 15. Analisar erros do modelo

In [ ]:
tabela_erros = tabelas["teste"][
    [
        "id",
        "texto",
        "perfil",
        "registro",
        "regiao",
    ]
].copy()

for indice, rotulo in enumerate(ROTULOS):
    tabela_erros[f"real_{rotulo}"] = (
        rotulos_teste[:, indice]
    )
    tabela_erros[f"prob_{rotulo}"] = (
        probabilidades_teste[:, indice]
    )
    tabela_erros[f"prev_{rotulo}"] = (
        previstos_otimizados[:, indice]
    )

tabela_erros["quantidade_erros"] = np.sum(
    rotulos_teste != previstos_otimizados,
    axis=1,
)

erros = tabela_erros[
    tabela_erros["quantidade_erros"] > 0
].sort_values(
    "quantidade_erros",
    ascending=False,
)

print("Exemplos com pelo menos um erro:", len(erros))

display(erros.head(30))

erros.to_csv(
    PASTA_METRICAS / "erros_teste.csv",
    index=False,
)

## 16. Testar frases manualmente

In [ ]:
def classificar_textos(
    textos: list[str],
) -> pd.DataFrame:
    modelo_avaliacao = treinador.model
    modelo_avaliacao.eval()

    entradas = tokenizador(
        textos,
        padding=True,
        truncation=True,
        max_length=configuracao.comprimento_maximo,
        return_tensors="pt",
    )

    dispositivo = next(
        modelo_avaliacao.parameters()
    ).device

    entradas = {
        chave: valor.to(dispositivo)
        for chave, valor in entradas.items()
    }

    with torch.no_grad():
        logits = modelo_avaliacao(
            **entradas
        ).logits

    probabilidades = torch.sigmoid(
        logits
    ).cpu().numpy()

    linhas = []

    for texto, probabilidades_texto in zip(
        textos,
        probabilidades,
    ):
        linha = {
            "texto": texto,
        }

        ativos = []

        for rotulo, probabilidade, limiar in zip(
            ROTULOS,
            probabilidades_texto,
            limiares_otimizados,
        ):
            linha[f"prob_{rotulo}"] = float(
                probabilidade
            )

            if probabilidade >= limiar:
                ativos.append(rotulo)

        linha["rotulos_previstos"] = ativos
        linhas.append(linha)

    return pd.DataFrame(linhas)


frases_teste = [
    "socorro ele está me batendo",
    "ele disse que vai me matar e está com uma faca",
    "comprei uma faca para cortar carne",
    "esse jogo está bom pra caralho",
    "mana me acode ele não solta meu braço",
    "o filme mostrou uma cena de tiro",
]

display(
    classificar_textos(frases_teste)
)